# Clean BERT fine-tuning — CMPE 586

Preparation and execution use the same versioned Python pipeline. Protocol: `docs/training-protocol.md`; observed calibration: `docs/training-calibration.md`.

This notebook prepares and verifies persistent inputs and the shared initialization. Preparation performs **zero training updates**. The full-training switch below defaults to **False**. Completed calibration is reused as a recorded reference; no calibration is launched by this notebook.


In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError("Select a Python 3.11/3.12 runtime; the observed Colab runtime was 2026.07.")
if version("torch") != "2.11.0+cu128":
    raise RuntimeError("Select the observed Colab 2026.07 GPU runtime. Keep its PyTorch installation.")

PROJECT = Path("/content/cmpe586-turkish-punctuation")
REPOSITORY = "https://github.com/kckagancan/cmpe586-turkish-punctuation.git"
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
else:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT, text=True).strip()
    if dirty:
        raise RuntimeError("The checkout has local changes; review them before updating.")
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT), "-c", str(PROJECT / "requirements/colab.constraints.txt")], check=True)
sys.path.insert(0, str(PROJECT / "src"))
print("Project commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip())


## Runtime and project storage

Observed environment: **Colab 2026.07 / Python 3.12.13 / Tesla T4 / PyTorch 2.11.0+cu128**. The constraints file records the main observed dependency versions; the preparation report also saves the actual environment and a full `pip freeze` snapshot.

There is one Drive connection cell. It requests authorization only when Drive is not already mounted. All persistent project writes use `MyDrive/cmpe586-turkish-punctuation`. A later runtime may request authorization again.


In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime.")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive", timeout_ms=600_000)
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not ready; preparation has not started.")

STORAGE_ROOT = Path("/content/drive/MyDrive/cmpe586-turkish-punctuation")
RUN_ROOT = STORAGE_ROOT / "training-v1"
INITIALIZATIONS = STORAGE_ROOT / "initializations/training-v1"
SEED = 586
RUN_TAG = "v1"
TRAIN_OUTPUT = RUN_ROOT / f"train-clean-seed-{SEED}-{RUN_TAG}"
READY_DIRECTORY = STORAGE_ROOT / "preparation/training-v1" / f"seed-{SEED}"
print("GPU:", torch.cuda.get_device_name(0))
print("Project storage:", STORAGE_ROOT)


## Prepare persistent assets — no training

The preparation command preserves any existing temporary calibration assets, verifies copied files, and prepares only train/validation. It checks every eligible encoding and saves/reloads the shared base-encoder initialization on CPU. It creates no optimizer, training update, validation score, or training checkpoint. Existing different asset directories are never overwritten.

The persistent initialization, prepared-input cache, calibration report, and preparation report survive runtime deletion. The repository checkout and local model-download cache remain temporary and can be recreated.


In [ ]:
subprocess.run([
    sys.executable, "scripts/prepare_training.py",
    "--storage-root", str(STORAGE_ROOT),
    "--reuse-storage-root", "/content/cmpe586-calibration",
    "--seed", str(SEED), "--download-model",
], cwd=PROJECT, check=True)


## Preparation status

This confirms assets and source hashes, not training convergence or interruption recovery. Full-epoch validation/selection and training-checkpoint resume have not been exercised yet.


In [ ]:
from turkish_punctuation.training_io import sha256

readiness = json.loads((READY_DIRECTORY / "readiness.json").read_text())
if any(sha256(PROJECT / name) != expected for name, expected in readiness["source_hashes"].items()):
    raise RuntimeError("Code/configuration changed after preparation; rerun preparation.")
BASE_COMMAND = [sys.executable, "scripts/train.py", "--config", "configs/training.json", "--seed", str(SEED), "--initialization-directory", str(INITIALIZATIONS), "--download-model"]
print(json.dumps({
    "stage": readiness["stage"],
    "training_started": readiness["training_started"],
    "optimizer_updates_in_preparation": readiness["optimizer_updates_in_preparation"],
    "train_records": readiness["inputs"]["train"]["records"],
    "validation_records": readiness["inputs"]["validation"]["records"],
    "initialization_tensors_sha256": readiness["initialization_tensors_sha256"],
    "calibration_reference": readiness["calibration_reference"],
    "persistent_storage": readiness["storage_root"],
}, indent=2))


## Full clean baseline — manual, disabled

Preparation ends above. This cell starts the three-epoch clean baseline only when `RUN_FULL_TRAINING=True`. It starts from the shared initialization, not from calibration-updated weights. For an interrupted formal run, keep identical settings/paths and use `RESUME=True`. A completed run is never overwritten.


In [ ]:
RUN_FULL_TRAINING = False
RESUME = False
if RUN_FULL_TRAINING:
    if readiness["storage_root"] != str(STORAGE_ROOT) or readiness["seed"] != SEED:
        raise RuntimeError("Storage/seed changed; rerun preparation.")
    if any(sha256(PROJECT / name) != expected for name, expected in readiness["source_hashes"].items()):
        raise RuntimeError("Code/configuration changed; rerun preparation.")
    if not RESUME and readiness["calibration_reference"] is None:
        raise RuntimeError("Review a matching GPU calibration before formal training.")
    command = [*BASE_COMMAND, "--mode", "train", "--output-directory", str(TRAIN_OUTPUT)]
    if RESUME:
        command.append("--resume")
    import signal

    # Stream child-process logs into the notebook while training is running.
    with subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            raise
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
else:
    print("Preparation only. Full training has not been started.")


## Saved outputs

Preparation: `preparation/training-v1/seed-586/{readiness.json,pip-freeze.txt}`. Input cache: `inputs/clean-v1/`. Shared initialization: `initializations/training-v1/seed-586/`.

Formal training later writes `run.json`, step events, rolling resumable checkpoints, epoch models and clean validation predictions/metrics under its own run directory. `summary.json` exists only after all epochs complete. No test scores are produced by this notebook.


In [ ]:
summary_path = TRAIN_OUTPUT / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Best epoch:", summary["best_epoch"])
    print("Clean validation macro-F1:", summary["best_macro_f1"])
    print("Best model:", TRAIN_OUTPUT / summary["best_model"])
else:
    print("No completed clean training result yet.")
